# ASG Airlines - Gold Layer

## Objective

The Gold layer creates a dimensional data warehouse from the Silver datasets.

Deliverables:
- Dimension tables
- Fact tables
- Business KPI views
- SQL-ready model for Power BI

Import Libraries

In [19]:
import pandas as pd
import sqlite3
from pathlib import Path

Read Silver Data

In [20]:
SILVER = Path("../data/silver")

flights = pd.read_parquet(SILVER / "flights.parquet")
passengers = pd.read_parquet(SILVER / "passengers.parquet")
bookings = pd.read_parquet(SILVER / "bookings.parquet")
payments = pd.read_parquet(SILVER / "payments.parquet")

print("Silver tables loaded.")

Silver tables loaded.


Create SQLite Database


In [21]:
DB_PATH = "../sql/airlines.db"

conn = sqlite3.connect(DB_PATH)

print("Database created successfully.")

Database created successfully.


Creating Dimension Tables (Flights , Passenger  )

Passenger Dimension

In [22]:
dim_passengers = passengers[
    [
        "passenger_key",
        "passenger_id",
        "first_name",
        "last_name",
        "age",
        "gender",
        "date_of_birth",
        "aadhaar_hash",
        "email_hash",
        "phone_masked"
    ]
]

dim_passengers.to_sql(
    "dim_passengers",
    conn,
    if_exists="replace",
    index=False
)

1000

Flights Dimension

In [23]:
dim_flights = flights[
    [
        "flight_key",
        "flight_id",
        "airline",
        "source",
        "destination",
        "departure_time",
        "arrival_time",
        "duration_minutes",
        "is_overnight"
    ]
]

dim_flights.to_sql(
    "dim_flights",
    conn,
    if_exists="replace",
    index=False
)

1005

Creating Fact Table (Booking , Payments)

Fact Bookings

In [24]:
fact_bookings = bookings.merge(
    passengers[["passenger_key","passenger_id"]],
    on="passenger_id",
    how="left"
).merge(
    flights[["flight_key","flight_id"]],
    on="flight_id",
    how="left"
)

fact_bookings = fact_bookings[
    [
        "booking_id",
        "passenger_key",
        "flight_key",
        "booking_date",
        "status",
        "seat_number"
    ]
]

fact_bookings.to_sql(
    "fact_bookings",
    conn,
    if_exists="replace",
    index=False
)

1002

Fact Payments

In [25]:
fact_payments = payments[
    [
        "payment_id",
        "booking_id",
        "amount",
        "payment_method"
    ]
]

fact_payments.to_sql(
    "fact_payments",
    conn,
    if_exists="replace",
    index=False
)

1000

Verifying the Tables

In [26]:
tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    conn
)

tables

,name
0,dim_passengers
1,dim_flights
2,fact_bookings
3,fact_payments


Connecting to SQLlite


In [27]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("../sql/airlines.db")

# Create / connect to SQLite database
conn = sqlite3.connect(DB_PATH)

print("Connected to:", DB_PATH)

Connected to: ..\sql\airlines.db


Load Silver Tables 

In [28]:
SILVER = Path("../data/silver")

flights = pd.read_parquet(SILVER / "flights.parquet")
passengers = pd.read_parquet(SILVER / "passengers.parquet")
bookings = pd.read_parquet(SILVER / "bookings.parquet")
payments = pd.read_parquet(SILVER / "payments.parquet")

print("Silver tables loaded successfully.")

Silver tables loaded successfully.


Dimension Table

In [29]:
dim_flights = flights[
    [
        "flight_key","flight_id","airline","source",
        "destination","departure_time","arrival_time",
        "duration_minutes","is_overnight"
    ]
]

dim_passengers = passengers[
    [
        "passenger_key","passenger_id","first_name","last_name",
        "age","gender","date_of_birth",
        "aadhaar_hash","email_hash","phone_masked"
    ]
]

dim_flights.to_sql("dim_flights", conn, if_exists="replace", index=False)
dim_passengers.to_sql("dim_passengers", conn, if_exists="replace", index=False)

print("Dimension tables created.")

Dimension tables created.


Fact Table 

In [30]:
fact_bookings = bookings.merge(
    passengers[["passenger_key","passenger_id"]],
    on="passenger_id",
    how="left"
).merge(
    flights[["flight_key","flight_id"]],
    on="flight_id",
    how="left"
)

fact_bookings = fact_bookings[
    [
        "booking_id","passenger_key","flight_key",
        "booking_date","status","seat_number"
    ]
]

fact_payments = payments[
    [
        "payment_id","booking_id","amount","payment_method"
    ]
]

fact_bookings.to_sql("fact_bookings", conn, if_exists="replace", index=False)
fact_payments.to_sql("fact_payments", conn, if_exists="replace", index=False)

print("Fact tables created.")

Fact tables created.


Verification

In [31]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)

,name
0,dim_flights
1,dim_passengers
2,fact_bookings
3,fact_payments


SQL Business KPIs

KPI 1: Flights by Airline

In [32]:
query = """
SELECT
    airline,
    COUNT(*) AS total_flights
FROM dim_flights
GROUP BY airline
ORDER BY total_flights DESC;
"""

kpi_airline = pd.read_sql(query, conn)
kpi_airline

,airline,total_flights
0,IndiGo,249
1,Air India,247
2,SpiceJet,241
3,Vistara,226
4,UNKNOWN,30
5,Indigo,12


KPI 2: Average Flight Duration

In [33]:
query = """
SELECT
    airline,
    ROUND(AVG(duration_minutes),2) AS avg_duration
FROM dim_flights
GROUP BY airline
ORDER BY avg_duration DESC;
"""

kpi_duration = pd.read_sql(query, conn)
kpi_duration

,airline,avg_duration
0,IndiGo,167.70
1,Air India,165.17
2,Vistara,163.67
3,SpiceJet,163.07
4,UNKNOWN,161.57
5,Indigo,134.50


KPI 3: Top 10 Routes

In [34]:
query = """
SELECT
    source,
    destination,
    COUNT(*) AS total_flights
FROM dim_flights
GROUP BY source, destination
ORDER BY total_flights DESC
LIMIT 10;
"""

kpi_routes = pd.read_sql(query, conn)
kpi_routes

,source,destination,total_flights
0,BOM,CCU,90
1,CCU,DEL,72
2,MAA,BLR,65
3,BLR,BOM,60
4,HYD,MAA,57
5,DEL,HYD,54
6,HYD,DEL,42
7,BOM,DEL,39
8,CCU,BOM,33
9,DEL,BLR,29


KPI 4: Booking Status

In [35]:
query = """
SELECT
    status,
    COUNT(*) AS bookings
FROM fact_bookings
GROUP BY status;
"""

kpi_status = pd.read_sql(query, conn)
kpi_status

,status,bookings
0,CANCELLED,315
1,CONFIRMED,320
2,INVALID,30
3,PENDING,292
4,UNKNOWN,45


KPI 5: Total Revenue

KPI CARD

In [36]:
query = """
SELECT
    ROUND(SUM(amount),2) AS total_revenue
FROM fact_payments;
"""

kpi_revenue = pd.read_sql(query, conn)
kpi_revenue

,total_revenue
0,7385142.98


KPI 6: Revenue by Airline

In [37]:
query = """
SELECT

    f.airline,

    ROUND(SUM(p.amount),2) AS revenue

FROM fact_payments p

JOIN fact_bookings b
ON p.booking_id = b.booking_id

JOIN dim_flights f
ON b.flight_key = f.flight_key

GROUP BY f.airline

ORDER BY revenue DESC;
"""

kpi_airline_revenue = pd.read_sql(query, conn)
kpi_airline_revenue

,airline,revenue
0,Vistara,2109039.78
1,SpiceJet,1860908.01
2,Air India,1684353.32
3,IndiGo,1477798.71
4,UNKNOWN,158487.58
5,Indigo,95792.32


KPI 7: Payment Method Split

In [38]:
query = """
SELECT

    payment_method,

    COUNT(*) AS transactions,

    ROUND(SUM(amount),2) AS revenue

FROM fact_payments

GROUP BY payment_method

ORDER BY revenue DESC;
"""

kpi_payment = pd.read_sql(query, conn)
kpi_payment

,payment_method,transactions,revenue
0,UPI,358,2618686.77
1,CARD,329,2400812.04
2,NETBANKING,313,2365644.17


KPI 8: Overnight Flights

In [39]:
query = """
SELECT

    CASE

        WHEN is_overnight = 1
        THEN 'Overnight'

        ELSE 'Same Day'

    END AS flight_type,

    COUNT(*) AS flights

FROM dim_flights

GROUP BY flight_type;
"""

kpi_overnight = pd.read_sql(query, conn)
kpi_overnight

,flight_type,flights
0,Overnight,122
1,Same Day,883


KPI 9: Passenger Age Groups

In [40]:
query = """
SELECT

CASE

WHEN age < 18 THEN 'Under 18'

WHEN age BETWEEN 18 AND 35 THEN '18-35'

WHEN age BETWEEN 36 AND 60 THEN '36-60'

ELSE '60+'

END AS age_group,

COUNT(*) AS passengers

FROM dim_passengers

GROUP BY age_group

ORDER BY passengers DESC;
"""

kpi_age = pd.read_sql(query, conn)
kpi_age

,age_group,passengers
0,60+,306
1,36-60,273
2,Under 18,215
3,18-35,206


KPI 10: Most Popular Destinations

In [41]:
query = """
SELECT

    destination,

    COUNT(*) AS flights

FROM dim_flights

GROUP BY destination

ORDER BY flights DESC;
"""

kpi_destination = pd.read_sql(query, conn)
kpi_destination

,destination,flights
0,DEL,198
1,CCU,187
2,BOM,169
3,BLR,163
4,MAA,147
5,HYD,141


Saving All Kpi Tables 

In [42]:
KPI = Path("../data/gold")
KPI.mkdir(parents=True, exist_ok=True)

tables = {
    "airline_flights": kpi_airline,
    "avg_duration": kpi_duration,
    "top_routes": kpi_routes,
    "booking_status": kpi_status,
    "revenue": kpi_revenue,
    "airline_revenue": kpi_airline_revenue,
    "payment_method": kpi_payment,
    "overnight": kpi_overnight,
    "age_group": kpi_age,
    "destination": kpi_destination
}

for name, df in tables.items():
    df.to_csv(KPI / f"{name}.csv", index=False)

print("Gold KPI datasets created successfully.")

Gold KPI datasets created successfully.


Final Summary 

# Gold Layer Summary

The Gold layer transformed the Silver datasets into business-ready analytical models.

## Deliverables

- 2 Dimension tables
- 2 Fact tables
- 10 SQL KPI datasets
- SQLite warehouse
- Star schema implementation

These datasets will be directly connected to Power BI for executive reporting.